In [ ]:
"""
complexity_figs.py -- regenerate the Dixon-vs-Groebner complexity figures.

Pure Python (no Sage).  Exact integer arithmetic throughout; only the final
log2 is floating point.  This script replaces the hard-coded parameters of
Complexity_Comparision.ipynb by an explicit figure registry, so that each
figure of the paper can be reproduced with the parameters stated in its
caption and written to the file name used by the TeX source:

    Section 3.3, Figure 1   -> complexity_comparison_d2.png     (d=2, w=2.81)
    Appendix E, Figure 6a   -> complexity_comparison_d5.png     (d=5, w=2.81)
    Appendix E, Figure 6b   -> complexity_vs_degree_n3.png      (n=3, w=2.81)
    Appendix E, Figure 6c   -> complexity_vs_degree_n5.png      (n=5, w=2.81)
    Appendix E, Figure 6d   -> complexity_vs_degree_n5w237.png  (n=5, w=2.37)

The last panel uses the max-over-stages Dixon model (see Appendix H) and is
therefore NOT produced here; it is generated by export_vector_figures.py using comp_step1&4.ipynb.

Usage:
    python3 complexity_figs.py            # regenerate Figure 1
    python3 complexity_figs.py --all      # regenerate the four plain panels
"""

import sys
from math import comb, log2, sqrt

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman", "DejaVu Serif", "Georgia"]
plt.rcParams["mathtext.fontset"] = "cm"


# ---------------------------------------------------------------------------
# big-integer safe log2
# ---------------------------------------------------------------------------

def flog2(x):
    """log2 of a positive (possibly huge) integer or float."""
    if isinstance(x, float):
        return log2(x)
    x = int(x)
    if x <= 0:
        return float("-inf")
    b = x.bit_length()
    if b <= 512:
        return log2(x)
    shift = b - 64
    return (b - 1) + log2((x >> shift) / float(1 << 63))


def binom(a, b):
    if b < 0 or a < 0 or b > a:
        return 0
    return comb(a, b)


# ---------------------------------------------------------------------------
# complexity models (all returned on the log2 scale)
# ---------------------------------------------------------------------------

def fuss_catalan(n, d):
    """C_n^(d) = binom(nd, n) / (n(d-1)+1)."""
    num, den = comb(n * d, n), n * (d - 1) + 1
    assert num % den == 0
    return num // den


def macaulay_bound(n, d):
    """Macaulay bound d_reg <= 1 + n(d-1)."""
    return 1 + n * (d - 1)


def dixon(n, d, omega):
    """Dixon resultant, well-determined case: n*d*C_n^(d)^omega."""
    return flog2(n * d) + omega * flog2(fuss_catalan(n, d))


def gb_classical(n, d, omega):
    """GB (classical) [Fau02]: binom(n+d_reg, d_reg)^omega."""
    dreg = macaulay_bound(n, d)
    return omega * flog2(binom(n + dreg, dreg))


def gb_f5(n, d, omega):
    """GB (F5) [BFS15,BBLP22]: n*d_reg*binom(n+d_reg-1, d_reg)^omega."""
    dreg = macaulay_bound(n, d)
    return flog2(n * dreg) + omega * flog2(binom(n + dreg - 1, dreg))


def gb_f5_refined(n, d, omega):
    """GB (F5 refined) [BFS15]: Macaulay row echelon form, rows*cols*rank^(w-2)."""
    dreg = macaulay_bound(n, d)
    n_cols = binom(n * d + 1, n)
    n_rows = n * binom(d * (n - 1) + 1, n)
    rank = n_cols - d ** n
    return flog2(n_rows) + flog2(n_cols) + (omega - 2) * flog2(rank)


def gb_per_degree(n, d, omega):
    """GB (per-deg) [Spa12,KLR24]: n * sum_i binom(n+i-d-1,i-d) binom(n+i-1,i)^(w-1)."""
    dsolv = macaulay_bound(n, d)
    total = 0.0
    for i in range(dsolv + 1):
        t = binom(n + i - d - 1, i - d)
        if t == 0:
            continue
        c = flog2(n * t) + (omega - 1) * flog2(binom(n + i - 1, i))
        total = c if total == 0.0 else max(total, c) + log2(
            1.0 + 2.0 ** (-abs(total - c)))
    return total


def fglm(n, d, omega):
    """FGLM [FGLM93]: n * d_I^omega with d_I = d^n."""
    return flog2(n) + omega * n * flog2(d)


def BNS_order_change(n, d, omega):
    """Change of order by Hermite normal form [BNS22]: t^(w-1) * d_I,
    with the generic-system asymptotics t = d^(n-1)/sqrt(n), d_I = d^n
    of [BNS22, Tbl. 1] (from [FM17, Cor. 5.10])."""
    log_t = (n - 1) * flog2(d) - 0.5 * flog2(n)
    return (omega - 1) * log_t + n * flog2(d)


MODELS = [
    ("Dixon Resultant", dixon,          "red"),
    ("GB (Classical)",  gb_classical,   "blue"),
    ("GB (F5)",         gb_f5,          "green"),
    ("GB (F5 refined)", gb_f5_refined,  "purple"),
    ("GB (per-deg)",    gb_per_degree,  "brown"),
    ("FGLM",            fglm,           "orange"),
    ("BNS order change",       BNS_order_change,        "darkcyan"),
]


# ---------------------------------------------------------------------------
# plotting
# ---------------------------------------------------------------------------

def plot_vs_n(fn, d, omega, n_values=range(3, 21)):
    plt.figure(figsize=(14, 9))
    for label, f, col in MODELS:
        ys = [f(n, d, omega) for n in n_values]
        plt.plot(list(n_values), ys, label=label, color=col, linestyle="-",
                 linewidth=2, marker="o", markersize=5)
    plt.xticks(range(min(n_values), max(n_values) + 1, 2))
    plt.xlabel("Number of Variables/Equations (n)", fontsize=26)
    plt.ylabel(r"log$_2$(Complexity)", fontsize=26)
    plt.title("Complexity Comparison: Dixon vs Gr\u00f6bner Basis Methods\n"
              f"(d={d}, \u03c9={omega:.2f})", fontsize=28, fontweight="bold")
    plt.legend(loc="best", fontsize=22)
    plt.grid(True, alpha=0.3)
    plt.tick_params(axis="both", which="major", labelsize=24)
    plt.tight_layout()
    plt.savefig(fn, dpi=300, bbox_inches="tight")
    plt.close()
    print("wrote", fn)


def plot_vs_d(fn, n, omega, d_values=range(2, 20)):
    plt.figure(figsize=(14, 9))
    for label, f, col in MODELS:
        ys = [f(n, d, omega) for d in d_values]
        plt.plot(list(d_values), ys, label=label, color=col, linestyle="-",
                 linewidth=2, marker="s", markersize=6)
    plt.xticks(range(min(d_values), max(d_values) + 1, 2))
    plt.xlabel("Polynomial Degree (d)", fontsize=24)
    plt.ylabel(r"log$_2$(Complexity)", fontsize=24)
    plt.title("Complexity vs Degree: Dixon vs Gr\u00f6bner Basis Methods\n"
              f"(n={n}, \u03c9={omega:.2f})", fontsize=26, fontweight="bold")
    plt.legend(loc="best", fontsize=20)
    plt.grid(True, alpha=0.3)
    plt.tick_params(axis="both", which="major", labelsize=22)
    plt.tight_layout()
    plt.savefig(fn, dpi=300, bbox_inches="tight")
    plt.close()
    print("wrote", fn)


def table(n_values, d, omega):
    print(f"\n{'n':>4} | " + " | ".join(f"{lab:>16}" for lab, _, _ in MODELS))
    for n in n_values:
        print(f"{n:>4} | " + " | ".join(
            f"{f(n, d, omega):>16.2f}" for _, f, _ in MODELS))


if __name__ == "__main__":
    plot_vs_n("complexity_comparison_d2.png", d=2, omega=2.81)
    table(range(3, 21), d=2, omega=2.81)
    # if "--all" in sys.argv:
    plot_vs_n("complexity_comparison_d5.png", d=5, omega=2.81)
    plot_vs_d("complexity_vs_degree_n3.png", n=3, omega=2.81)
    plot_vs_d("complexity_vs_degree_n5.png", n=5, omega=2.81)
    from export_vector_figures import export_fig6d
    export_fig6d()